In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
)
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
 
warnings.filterwarnings("ignore")
np.random.seed(42)

1. GENERATE SYNTHETIC DATA

In [21]:
N = 500
 
print("=" * 55)
print("  Crop Health Prediction — XGBoost Test Pipeline")
print("=" * 55)
print(f"\n[1] Generating synthetic dataset ({N} rows, 5 features)...")
 
NDVI                  = np.random.uniform(0.0, 1.0, N)
SAVI                  = np.random.uniform(0.0, 1.0, N)
Chlorophyll_Content   = np.random.uniform(10, 80, N)
Leaf_Area_Index       = np.random.uniform(0.5, 8.0, N)
Crop_Stress_Indicator = np.random.randint(0, 101, N)
 
# Target: realistic rules — not pure random
# Healthy when NDVI high, stress low, chlorophyll high
health_score = (
    (NDVI > 0.5).astype(int) * 2                    # strongest signal
    + (Crop_Stress_Indicator < 40).astype(int) * 2  # direct stress flag
    + (Chlorophyll_Content > 35).astype(int)         # plant vigor
    + (Leaf_Area_Index > 2.0).astype(int)            # canopy density
    + (SAVI > 0.4).astype(int)                       # soil-corrected health
)
# score >= 4 → healthy (~70% of rows, matching dataset distribution)
Crop_Health_Label = (health_score >= 4).astype(int)
 
df = pd.DataFrame({
    "NDVI":                  NDVI,
    "SAVI":                  SAVI,
    "Chlorophyll_Content":   Chlorophyll_Content,
    "Leaf_Area_Index":       Leaf_Area_Index,
    "Crop_Stress_Indicator": Crop_Stress_Indicator,
    "Crop_Health_Label":     Crop_Health_Label,
})
 
print(f"    Shape: {df.shape}")
print(f"    Healthy (1): {Crop_Health_Label.sum()} | Unhealthy (0): {(Crop_Health_Label == 0).sum()}")
print(f"\n    Sample rows:")
print(df.head(5).to_string(index=False))

  Crop Health Prediction — XGBoost Test Pipeline

[1] Generating synthetic dataset (500 rows, 5 features)...
    Shape: (500, 6)
    Healthy (1): 296 | Unhealthy (0): 204

    Sample rows:
    NDVI     SAVI  Chlorophyll_Content  Leaf_Area_Index  Crop_Stress_Indicator  Crop_Health_Label
0.374540 0.698162            22.959305         4.393113                     91                  0
0.950714 0.536096            47.933066         4.093864                      6                  1
0.731994 0.309528            71.106209         0.692315                     34                  1
0.598658 0.813795            61.255742         3.059359                     54                  1
0.156019 0.684731            66.459280         3.351467                     91                  0


2. SPLIT FEATURES AND TARGET

In [23]:
print("\n[2] Preparing features and target...")
 
FEATURE_COLS = [
    "NDVI",
    "SAVI",
    "Chlorophyll_Content",
    "Leaf_Area_Index",
    "Crop_Stress_Indicator",
]
X = df[FEATURE_COLS]
y = df["Crop_Health_Label"]
 
print(f"    Features (X): {FEATURE_COLS}")
print(f"    Target  (y): Crop_Health_Label")
 


[2] Preparing features and target...
    Features (X): ['NDVI', 'SAVI', 'Chlorophyll_Content', 'Leaf_Area_Index', 'Crop_Stress_Indicator']
    Target  (y): Crop_Health_Label


3. TRAIN / TEST SPLIT  (80% / 20%)

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n[3] Split → Train: {len(X_train)} rows | Test: {len(X_test)} rows")



[3] Split → Train: 400 rows | Test: 100 rows


4. HANDLE CLASS IMBALANCE (70/30)

In [25]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos
print(f"\n[4] scale_pos_weight = {n_neg}/{n_pos} = {scale_pos_weight:.2f}")


[4] scale_pos_weight = 163/237 = 0.69


5. TRAIN XGBOOST

In [28]:
print("\n[5] Training XGBoost...")
 
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    verbosity=0,
)
 
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False,
)
print("    Done.")



[5] Training XGBoost...
    Done.


6. EVALUATE

In [32]:
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]
auc         = roc_auc_score(y_test, y_pred_prob)
 
print(f"\n[6] Results:")
print(f"    ROC-AUC: {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Unhealthy (0)", "Healthy (1)"]))
print(confusion_matrix(y_test, y_pred))


[6] Results:
    ROC-AUC: 0.9988

               precision    recall  f1-score   support

Unhealthy (0)       0.97      0.93      0.95        41
  Healthy (1)       0.95      0.98      0.97        59

     accuracy                           0.96       100
    macro avg       0.96      0.95      0.96       100
 weighted avg       0.96      0.96      0.96       100

[[38  3]
 [ 1 58]]
